In [ ]:
from datasets import load_dataset
from sympy.physics.units import temperature

data = load_dataset("cornell-movie-review-data/rotten_tomatoes")
data

In [2]:
from transformers import pipeline

model_path = "cardiffnlp/twitter-roberta-base-sentiment-latest"

pipe = pipeline(
    model=model_path,
    tokenizer=model_path,
    top_k=None,
    device="cuda:0"
)



D:\python projects\hands_on_lmm\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 28733.98it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
import numpy as np
from tqdm import tqdm
from transformers.pipelines.pt_utils import KeyDataset

y_pred = []
for output in tqdm(pipe(KeyDataset(data["test"], "text")), total=len(data["test"])):
    scores = {entry["label"]: entry["score"] for entry in output}
    negative_score = scores["negative"]
    positive_score = scores["positive"]
    assignment = np.argmax([negative_score, positive_score])
    y_pred.append(assignment)

from sklearn.metrics import classification_report

def evaluate_performance(y_true, y_pred):
    performance = classification_report(y_true, y_pred, target_names=["negative review", "positive review"])
    print(performance)

evaluate_performance(data["test"]["label"], y_pred)

100%|██████████| 1066/1066 [00:10<00:00, 97.11it/s] 


                 precision    recall  f1-score   support

negative review       0.76      0.88      0.81       533
positive review       0.86      0.72      0.78       533

       accuracy                           0.80      1066
      macro avg       0.81      0.80      0.80      1066
   weighted avg       0.81      0.80      0.80      1066



In [13]:
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")

pipe = pipeline(
    task="text-classification",
    model=model,
    tokenizer=tokenizer,
    top_k=None,
    device="cuda:0"
)

y_pred = []
for output in tqdm(pipe(KeyDataset(data["test"], "text")), total=len(data["test"])):
    scores = {entry["label"]: entry["score"] for entry in output}
    negative_score = scores["NEGATIVE"]
    positive_score = scores["POSITIVE"]
    assignment = np.argmax([negative_score, positive_score])
    y_pred.append(assignment)

from sklearn.metrics import classification_report

evaluate_performance(data["test"]["label"], y_pred)

100%|██████████| 1066/1066 [00:05<00:00, 184.16it/s]


                 precision    recall  f1-score   support

negative review       0.89      0.90      0.90       533
positive review       0.90      0.89      0.90       533

       accuracy                           0.90      1066
      macro avg       0.90      0.90      0.90      1066
   weighted avg       0.90      0.90      0.90      1066



In [14]:
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression

model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

train_embeddings = model.encode(data["train"]["text"], show_progress_bar=True)
test_embeddings = model.encode(data["test"]["text"], show_progress_bar=True)
train_embeddings.shape

clf = LogisticRegression(random_state=42)
clf.fit(train_embeddings, data["train"]["label"])

y_pred = clf.predict(test_embeddings)
evaluate_performance(data["test"]["label"], y_pred)

Batches: 100%|██████████| 34/34 [00:01<00:00, 17.05it/s]


                 precision    recall  f1-score   support

negative review       0.85      0.86      0.85       533
positive review       0.86      0.85      0.85       533

       accuracy                           0.85      1066
      macro avg       0.85      0.85      0.85      1066
   weighted avg       0.85      0.85      0.85      1066



In [16]:
### zero-shot classification

from sklearn.metrics.pairwise import cosine_similarity

label_embeddings = model.encode(["A very negative review", "A very positive review"])

sim_matrix = cosine_similarity(test_embeddings, label_embeddings)
y_pred = np.argmax(sim_matrix, axis=1)

evaluate_performance(data["test"]["label"], y_pred)


                 precision    recall  f1-score   support

negative review       0.73      0.84      0.78       533
positive review       0.81      0.69      0.75       533

       accuracy                           0.76      1066
      macro avg       0.77      0.76      0.76      1066
   weighted avg       0.77      0.76      0.76      1066



In [6]:
import openai
import getpass

api_key = getpass.getpass("Enter your API key: ")
client = openai.OpenAI(api_key=api_key)

def chatgpt_generation(prompt, document, model="gpt-3.5-turbo-0125"):
   messages=[
        {
        "role": "system",
        "content": "You are a helpful assistant."
        },
        {
        "role": "user",
        "content": prompt.replace("[DOCUMENT]", document)
        }
    ]

   chat_completion = client.chat.completions.create(
       messages=messages,
       model=model,
       temperature=0
   )
   return chat_completion.choices[0].message.content

prompt = """Predict whether the following document is a positive or negative
movie review:
[DOCUMENT]
If it is positive return 1 and if it is negative return 0. Do not give any
other answers.
"""
# Predict the target using GPT
document = "unpretentious , charming , quirky , original"

chatgpt_generation(prompt, document)

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}